# Motivational Plots

## 0. Imports & Set-Up:

In [ ]:
from pathlib import Path


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans


In [ ]:
RANDOM_SEED = 42


In [7]:
from scipy.optimize import linear_sum_assignment


def align_cluster_labels(
    reference_labels: np.ndarray,
    labels: np.ndarray,
) -> np.ndarray:
    """
    Reorder labels to optimally match reference_labels using Hungarian assignment.
    """
    # Create common label space
    all_classes = np.union1d(reference_labels, labels)

    # Create square contingency matrix covering all class labels
    cont = np.zeros((len(all_classes), len(all_classes)), dtype=int)

    ref_to_idx = {val: i for i, val in enumerate(all_classes)}
    lab_to_idx = {val: i for i, val in enumerate(all_classes)}

    for r, l in zip(reference_labels, labels):
        cont[ref_to_idx[r], lab_to_idx[l]] += 1

    # Solve linear sum assignment
    row_ind, col_ind = linear_sum_assignment(-cont)

    # row_ind: indices in all_classes for reference
    # col_ind: indices in all_classes for predicted labels
    # Want to map: predicted class (at col_ind) -> reference class (at row_ind)
    mapping = {all_classes[c]: all_classes[r] for r, c in zip(row_ind, col_ind)}

    # Map original labels using 1-to-1 dictionary
    aligned = np.array([mapping[lbl] for lbl in labels], dtype=reference_labels.dtype)
    return aligned

In [9]:
%env PYTHONWARNINGS=ignore::FutureWarning,ignore::DeprecationWarning,ignore::UserWarning,ignore::RuntimeWarning

env: PYTHONWARNINGS=ignore::FutureWarning,ignore::DeprecationWarning,ignore::UserWarning,ignore::RuntimeWarning


---

## 1. Pre-Process the Data

### 1.1 Levine:

In [ ]:
from benchmarks.datasets import load_levine32

X_levine, y_levine, meta_levine = load_levine32()


### 1.2 Klein:

In [ ]:
from benchmarks.datasets import load_klein

X_klein, y_klein, meta_klein = load_klein(root=Path("../../data"))


---

## 3. Motivational Plots

In [53]:
levine_clusters_6 = KMeans(n_clusters=6, random_state=RANDOM_SEED).fit_predict(X_levine)
levine_clusters_10 = KMeans(n_clusters=10, random_state=RANDOM_SEED).fit_predict(
    X_levine
)
levine_clusters_14 = KMeans(n_clusters=14, random_state=RANDOM_SEED).fit_predict(
    X_levine
)
levine_clusters_18 = KMeans(n_clusters=18, random_state=RANDOM_SEED).fit_predict(
    X_levine
)

klein_clusters_2 = KMeans(n_clusters=2, random_state=RANDOM_SEED).fit_predict(X_klein)
klein_clusters_3 = KMeans(n_clusters=3, random_state=RANDOM_SEED).fit_predict(X_klein)
klein_clusters_4 = KMeans(n_clusters=4, random_state=RANDOM_SEED).fit_predict(X_klein)
klein_clusters_5 = KMeans(n_clusters=5, random_state=RANDOM_SEED).fit_predict(X_klein)

In [56]:
levine_clusters_6 = align_cluster_labels(levine_clusters_14, levine_clusters_6)
levine_clusters_10 = align_cluster_labels(levine_clusters_14, levine_clusters_10)
levine_clusters_18 = align_cluster_labels(levine_clusters_14, levine_clusters_18)

klein_clusters_2 = align_cluster_labels(klein_clusters_4, klein_clusters_2)
klein_clusters_3 = align_cluster_labels(klein_clusters_4, klein_clusters_3)
klein_clusters_5 = align_cluster_labels(klein_clusters_4, klein_clusters_5)

In [57]:
np.unique(klein_clusters_4)

array([0, 1, 2, 3], dtype=int32)

In [58]:
label_map = {0: 3, 1: 2, 2: 0, 3: 1, 4: 4}

klein_clusters_2 = np.array([label_map.get(lbl, lbl) for lbl in klein_clusters_2])
klein_clusters_3 = np.array([label_map.get(lbl, lbl) for lbl in klein_clusters_3])
klein_clusters_4 = np.array([label_map.get(lbl, lbl) for lbl in klein_clusters_4])
klein_clusters_5 = np.array([label_map.get(lbl, lbl) for lbl in klein_clusters_5])

In [59]:
np.unique(klein_clusters_3)

array([0, 1, 3])

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from benchmarks._panels import scatter_clusters
from benchmarks.figures import VIS_ROOT

ROW_LABEL_FS = 22
PANEL_LETTER_FS = 28

levine_embedding = TSNE(random_state=RANDOM_SEED).fit_transform(X_levine)
klein_embedding = PCA(n_components=2, random_state=RANDOM_SEED).fit_transform(X_klein)

fig, axes = plt.subplots(2, 4, figsize=(16, 9), constrained_layout=True)

levine_panels = [
    (6, levine_clusters_6),
    (10, levine_clusters_10),
    (14, levine_clusters_14),
    (18, levine_clusters_18),
]
for j, (k, lbl) in enumerate(levine_panels):
    scatter_clusters(
        axes[0, j],
        levine_embedding,
        lbl,
        title=f"k={k}",
        s=12,
    )

klein_panels = [
    (2, klein_clusters_2),
    (3, klein_clusters_3),
    (4, klein_clusters_4),
    (5, klein_clusters_5),
]
for j, (k, lbl) in enumerate(klein_panels):
    scatter_clusters(
        axes[1, j],
        klein_embedding,
        lbl,
        title=f"k={k}",
        s=30,
    )

# Row labels (dataset name) on leftmost column. scatter_clusters hides
# spines and ticks but leaves ylabel rendering intact, so we set it after
# the call.
axes[0, 0].set_ylabel("Levine", fontsize=ROW_LABEL_FS, fontweight="bold", labelpad=14)
axes[1, 0].set_ylabel("Klein", fontsize=ROW_LABEL_FS, fontweight="bold", labelpad=14)

# Panel letters A / B at the top-left corner of each row.
fig.text(
    0.005, 0.965, "A", fontsize=PANEL_LETTER_FS, fontweight="bold", va="top", ha="left"
)
fig.text(
    0.005, 0.485, "B", fontsize=PANEL_LETTER_FS, fontweight="bold", va="top", ha="left"
)

# The historical location, resolved against the repo root the same way
# every other figure in this package is. Copying the result into the
# manuscript tree stays a manual step.
OUT_PATH = VIS_ROOT / "motivational_plots" / "clustering_problem.png"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT_PATH, dpi=300, bbox_inches="tight")
print(f"saved figure to {OUT_PATH.resolve()}")
plt.show()


---